In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
df = pd.read_csv(r"C:\Users\ANKIT\Downloads\cardekho_imputated.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [4]:
## Removing unncessary columns

df.drop('Unnamed: 0', axis=1, inplace=True)
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [5]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [6]:
## Numeric Feature
num_feature = [feature for feature in df.columns if df[feature].dtype!= 'O']
print('Numerical Feature :',len(num_feature))

## Categorical Feature
cat_feature = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Categorical Feature :',len(cat_feature))

## Discrete Feature
dis_feature = [feature for feature in num_feature if len(df[feature].unique()) <= 25]
print('Discrete Feature :',len(dis_feature))

## Continuous Feature
con_feature = [feature for feature in num_feature if feature not in dis_feature]
print('Continuous Feature :',len(con_feature))

Numerical Feature : 7
Categorical Feature : 4
Discrete Feature : 2
Continuous Feature : 5


## Independent and Dependent Feature

In [7]:
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'],axis = 1)
y = df['selling_price']

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [9]:
## Create Column Transformer with 3 types of transformer
num_featues = X.select_dtypes(exclude = 'object').columns
onehot_columns = ['seller_type','fuel_type','transmission_type']
label_encoder_columns = ['model']

In [10]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop = 'first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder',oh_transformer,onehot_columns),
        ('StandardScaler',numeric_transformer,num_featues)
    ],remainder = 'passthrough'
)


In [11]:
X = preprocessor.fit_transform(X)

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 10)

## Model Training and Model Selection

In [14]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [15]:
## Create a function to evaluate the model
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mean_squared_error(true,predicted))
    r2_square = r2_score(true,predicted)
    return mae,mse,rmse,r2_square

In [16]:
## Beginning Model Training
models = {
    'Ada Boost' : AdaBoostRegressor(),
    'Linear Regression' : LinearRegression(),
    'Lasso': Lasso(),
    'Ridge' : Ridge(),
    'XGB' : XGBRegressor(),
    'K - Neighbors Regresssor' : KNeighborsRegressor(),
    'Decision Tree' : DecisionTreeRegressor(),
    'Random Foresst' : RandomForestRegressor()
}

In [17]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)# Train the Model
    ## Make Prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Evalaute Train and test data
    model_train_mae,model_train_mse,model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)

    model_test_mae,model_test_mse,model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance for Training Set')
    print('-Mean Absolute Error : {:.4f}'.format(model_train_mae))
    print('-Mean Squared Error : {:.4f}'.format(model_train_mse))
    print('-Root Mean Absolute Error : {:.4f}'.format(model_train_rmse))
    print('-R2 Score Error : {:.4f}'.format(model_train_r2))
    print('\n')
    print('-----------------------------------------------------------')
    print('\n')
    print('Model Performance for Test Set')
    print('-Mean Absolute Error : {:.4f}'.format(model_test_mae))
    print('-Mean Squared Error : {:.4f}'.format(model_test_mse))
    print('-Root Mean Absolute Error : {:.4f}'.format(model_test_rmse))
    print('-R2 Score Error : {:.4f}'.format(model_test_r2))
    print('\n')
    print('='*62)

Ada Boost
Model Performance for Training Set
-Mean Absolute Error : 307570.3441
-Mean Squared Error : 173875491161.2439
-Root Mean Absolute Error : 416983.8020
-R2 Score Error : 0.7545


-----------------------------------------------------------


Model Performance for Test Set
-Mean Absolute Error : 314604.8762
-Mean Squared Error : 437374595051.1369
-Root Mean Absolute Error : 661343.0237
-R2 Score Error : 0.6242


Linear Regression
Model Performance for Training Set
-Mean Absolute Error : 260031.9512
-Mean Squared Error : 229251209296.3019
-Root Mean Absolute Error : 478801.8476
-R2 Score Error : 0.6763


-----------------------------------------------------------


Model Performance for Test Set
-Mean Absolute Error : 280915.9804
-Mean Squared Error : 567633907510.6738
-Root Mean Absolute Error : 753414.8310
-R2 Score Error : 0.5123


Lasso
Model Performance for Training Set
-Mean Absolute Error : 260030.0543
-Mean Squared Error : 229251214150.1726
-Root Mean Absolute Error : 4788

In [18]:
## Initialize few parameter for hyperparameter tuning

xgb_params = {
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "n_estimators": [200, 500, 800, 1200],
    "max_depth": [3, 5, 7, 10],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 5],
    "reg_lambda": [1, 5, 10]
}

In [19]:
## Models list for hyperparameter tuning

randomcv_models = [('XGB',XGBRegressor(),xgb_params)]

In [20]:
from sklearn.model_selection import RandomizedSearchCV
model_param = {}
for name,model,params in randomcv_models:
    random = RandomizedSearchCV(estimator = model,
                               param_distributions = params,
                               n_iter = 100,
                               cv = 3,
                               verbose = 2,
                               n_jobs = -1)

    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f'-----------------------Best Params for {model_name}--------------------')
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
-----------------------Best Params for XGB--------------------
{'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 5, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0.3, 'colsample_bytree': 0.8}


In [21]:
## Restraing the model with best parameter
models = {

    'XGB Regressor' : XGBRegressor(subsample = 0.8, 
                                   learning_rate = 0.2,
                                   reg_lambda = 10,
                                    reg_alpha = 5,
                                    n_estimators = 500,
                                    min_child_weight = 5,
                                    max_depth = 5,
                                    gamma = 0.3,
                                    colsample_bytree = 0.8
                                           )
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)# Train the Model
    ## Make Prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Evalaute Train and test data
    model_train_mae,model_train_mse,model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)

    model_test_mae,model_test_mse,model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance for Training Set')
    print('-Mean Absolute Error : {:.4f}'.format(model_train_mae))
    print('-Mean Squared Error : {:.4f}'.format(model_train_mse))
    print('-Root Mean Absolute Error : {:.4f}'.format(model_train_rmse))
    print('-R2 Score Error : {:.4f}'.format(model_train_r2))
    print('\n')
    print('-----------------------------------------------------------')
    print('\n')
    print('Model Performance for Test Set')
    print('-Mean Absolute Error : {:.4f}'.format(model_test_mae))
    print('-Mean Squared Error : {:.4f}'.format(model_test_mse))
    print('-Root Mean Absolute Error : {:.4f}'.format(model_test_rmse))
    print('-R2 Score Error : {:.4f}'.format(model_test_r2))
    print('\n')
    print('='*62)

XGB Regressor
Model Performance for Training Set
-Mean Absolute Error : 67071.3984
-Mean Squared Error : 9334608896.0000
-Root Mean Absolute Error : 96615.7797
-R2 Score Error : 0.9868


-----------------------------------------------------------


Model Performance for Test Set
-Mean Absolute Error : 103257.0156
-Mean Squared Error : 283653767168.0000
-Root Mean Absolute Error : 532591.5575
-R2 Score Error : 0.7563


